In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as data
import pickle
import pandas as pd
import time
import os
import scipy.sparse as sp
import shutil
import random

# ---------------- CONFIGURATION ----------------
class Args:
    def __init__(self):
        self.lr = 1e-3
        self.decay = 0.99
        self.batch = 256
        self.lambda1 = 1e-7  
        self.epoch = 100     
        self.d = 32         # From old_setting (was 64)
        self.q = 5
        self.gnn_layer = 2
        self.data = 'yelp'
        self.dropout = 0.25 # From old_setting (was 0.0)
        self.temp = 0.5     # From old_setting (was 0.2)
        self.lambda2 = 1e-4 # From old_setting (was 1e-7)
        self.cuda = '0'
        # MSBEGCL parameters
        self.msb_rate = 0.1  # gamma (Weight for l_sim)
        self.sim_threshold = 0.1 # Similarity threshold

args = Args()

if torch.cuda.is_available():
    device = 'cuda:' + args.cuda
else:
    device = 'cpu'

print(f"Using device: {device}")
os.environ["TORCH_AUTOGRAD_SHUTDOWN_WAIT_LIMIT"] = "0"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# ---------------- DATA SETUP ----------------
data_path = 'data/' + args.data + '/'
if not os.path.exists(data_path):
    print(f"Data folder '{data_path}' not found. Downloading from GitHub...")
    try:
        if os.path.exists('temp_LightGCL_repo'):
            shutil.rmtree('temp_LightGCL_repo')
        
        # Clone the repository
        os.system('git clone https://github.com/HKUDS/LightGCL.git temp_LightGCL_repo')
        
        # Move data folder
        if os.path.exists('temp_LightGCL_repo/data'):
            if not os.path.exists('data'):
                shutil.copytree('temp_LightGCL_repo/data', 'data')
            else:
                if not os.path.exists(data_path):
                    target_dir = os.path.dirname(data_path.rstrip('/'))
                    if not os.path.exists(target_dir):
                        os.makedirs(target_dir)
                    # Copy specific dataset if missing
                    # Note: git clone gives data/yelp/trnMat.pkl
                    # We want to move temp_LightGCL_repo/data/yelp to data/yelp
                    src_path = f'temp_LightGCL_repo/{data_path}'
                    if os.path.exists(src_path):
                         # If data/yelp does not exist
                         if not os.path.exists(data_path):
                             shutil.move(src_path, data_path)
            
            print("Data successfully downloaded and extracted.")
        else:
            print("Warning: Data extracted but 'data' folder not found in repo.")
            
    except Exception as e:
        print(f"Error downloading data: {e}. Please ensure internet access is enabled.")
    finally:
        # Cleanup
        if os.path.exists('temp_LightGCL_repo'):
            shutil.rmtree('temp_LightGCL_repo')

# ---------------- UTILS ----------------
def metrics(uids, predictions, topk, test_labels):
    user_num = 0
    all_recall = 0
    all_ndcg = 0
    for i in range(len(uids)):
        uid = uids[i]
        prediction = list(predictions[i][:topk])
        label = test_labels[uid]
        if len(label)>0:
            hit = 0
            idcg = np.sum([np.reciprocal(np.log2(loc + 2)) for loc in range(min(topk, len(label)))])
            dcg = 0
            for item in label:
                if item in prediction:
                    hit+=1
                    loc = prediction.index(item)
                    dcg = dcg + np.reciprocal(np.log2(loc+2))
            all_recall = all_recall + hit/len(label)
            all_ndcg = all_ndcg + dcg/idcg
            user_num+=1
    return all_recall/user_num, all_ndcg/user_num

def scipy_sparse_mat_to_torch_sparse_tensor(sparse_mx):
    sparse_mx = sparse_mx.tocoo().astype(np.float32)
    indices = torch.from_numpy(
        np.vstack((sparse_mx.row, sparse_mx.col)).astype(np.int64))
    values = torch.from_numpy(sparse_mx.data)
    shape = torch.Size(sparse_mx.shape)
    return torch.sparse_coo_tensor(indices, values, shape)

def sparse_dropout(mat, dropout):
    if dropout == 0.0:
        return mat
    indices = mat.indices()
    values = nn.functional.dropout(mat.values(), p=dropout)
    size = mat.size()
    return torch.sparse_coo_tensor(indices, values, size)

def spmm(sp, emb, device):
    sp = sp.coalesce()
    cols = sp.indices()[1]
    rows = sp.indices()[0]
    col_segs =  emb[cols] * torch.unsqueeze(sp.values(),dim=1)
    result = torch.zeros((sp.shape[0],emb.shape[1])).to(torch.device(device)) 
    result.index_add_(0, rows, col_segs)
    return result

def InfoNCE(view1, view2, temperature):
    view1, view2 = F.normalize(view1, dim=1), F.normalize(view2, dim=1)
    pos_score = (view1 * view2).sum(dim=-1)
    pos_score = torch.exp(pos_score / temperature)
    ttl_score = torch.matmul(view1, view2.transpose(0, 1))
    ttl_score = torch.exp(ttl_score / temperature).sum(dim=1)
    loss = -torch.log(pos_score / ttl_score + 1e-8)
    return torch.mean(loss)

def load_msbe_neighbors(interaction_mat, sim_threshold=0.1):
    """
    Compute Global Top-1 Similarity Neighbors.
    Uses Sparse Matrix Multiplication batch-wise for global similarity search.
    """
    print(f'Calculating Global Top-1 Neighbors (Threshold={sim_threshold})...')
    
    # Interaction Mat is user x item
    
    def get_global_top1(mat, threshold):
        # Normalize rows
        mat = mat.tocsr()
        row_sums = np.array(mat.power(2).sum(axis=1)).flatten()
        row_norms = np.sqrt(row_sums)
        row_norms[row_norms == 0] = 1.0
        diag_norm = sp.diags(1.0 / row_norms)
        norm_mat = diag_norm.dot(mat)
        
        num_nodes = norm_mat.shape[0]
        neighbors = {}
        
        # Batch processing
        batch_size = 2000 
        
        for start_idx in range(0, num_nodes, batch_size):
            end_idx = min(start_idx + batch_size, num_nodes)
            
            # Batch: (B, F)
            batch_mat = norm_mat[start_idx:end_idx]
            
            # Sim: (B, N)
            sim_batch = batch_mat.dot(norm_mat.T)
            
            # Convert to dense to find max
            # Note: For large datasets, this dense matrix might be large (B x N)
            # 2000 x 30000 (users) floats = 60MB, acceptable.
            sim_dense = sim_batch.toarray()
            
            # Mask self
            for i in range(len(sim_dense)):
                global_id = start_idx + i
                sim_dense[i, global_id] = -1.0
            
            # Find max
            max_indices = np.argmax(sim_dense, axis=1)
            max_values = np.max(sim_dense, axis=1)
            
            for i in range(len(sim_dense)):
                if max_values[i] > threshold:
                    global_u = start_idx + i
                    best_n = max_indices[i]
                    neighbors[global_u] = [int(best_n)]
                    
            if (start_idx // batch_size) % 5 == 0:
                print(f'Processed {end_idx}/{num_nodes} nodes...')
                
        return neighbors

    # User Neighbors
    user_neighbors = get_global_top1(interaction_mat, sim_threshold)
    print(f'Found {len(user_neighbors)} users with similar neighbors > {sim_threshold}')
    
    # Item Neighbors (Transpose interaction matrix)
    item_neighbors = get_global_top1(interaction_mat.T, sim_threshold)
    print(f'Found {len(item_neighbors)} items with similar neighbors > {sim_threshold}')
    
    return user_neighbors, item_neighbors

# ---------------- MODEL ----------------
class W_contrastive(nn.Module):
    def __init__(self,d):
        super().__init__()
        self.W = nn.Parameter(nn.init.xavier_uniform_(torch.empty(d,d)))

    def forward(self,x):
        return x @ self.W

class LightGCL(nn.Module):
    def __init__(self, n_u, n_i, d, u_mul_s, v_mul_s, ut, vt, train_csr, adj_norm, l, temp, lambda_1, dropout, batch_user, device, user_neighbors=None, item_neighbors=None, msb_rate=0.0):
        super(LightGCL,self).__init__()
        self.E_u_0 = nn.Parameter(nn.init.xavier_uniform_(torch.empty(n_u,d)))
        self.E_i_0 = nn.Parameter(nn.init.xavier_uniform_(torch.empty(n_i,d)))

        self.train_csr = train_csr
        self.adj_norm = adj_norm
        self.l = l
        self.E_u_list = [None] * (l+1)
        self.E_i_list = [None] * (l+1)
        self.E_u_list[0] = self.E_u_0
        self.E_i_list[0] = self.E_i_0
        self.Z_u_list = [None] * (l+1)
        self.Z_i_list = [None] * (l+1)
        self.G_u_list = [None] * (l+1)
        self.G_i_list = [None] * (l+1)
        self.temp = temp
        self.tau = 2 # Fixed tau for MSBE loss
        self.lambda_1 = lambda_1
        self.dropout = dropout
        self.act = nn.LeakyReLU(0.5)
        self.batch_user = batch_user
        self.Ws = nn.ModuleList([W_contrastive(d) for i in range(l)])

        self.E_u = None
        self.E_i = None

        self.u_mul_s = u_mul_s
        self.v_mul_s = v_mul_s
        self.ut = ut
        self.vt = vt

        self.device = device
        
        # MSBEGCL specific
        self.user_neighbors = user_neighbors if user_neighbors is not None else {}
        self.item_neighbors = item_neighbors if item_neighbors is not None else {}
        self.msb_rate = msb_rate

    def get_msb_samples(self, indices, neighbor_dict):
        res = []
        for i in indices:
            i = int(i)
            if i in neighbor_dict and len(neighbor_dict[i]) > 0:
                res.append(random.choice(neighbor_dict[i]))
            else:
                res.append(i) # Self-loop if no neighbor
        return torch.tensor(res).long().to(self.device)

    def forward(self, uids, iids, pos, neg, test=False):
        if test==True:  # testing phase
            preds = self.E_u[uids] @ self.E_i.T
            mask = self.train_csr[uids.cpu().numpy()].toarray()
            mask = torch.Tensor(mask).to(self.device)
            preds = preds * (1-mask)
            predictions = preds.argsort(descending=True)
            return predictions
        else:  # training phase
            for layer in range(1,self.l+1):
                # GNN propagation
                self.Z_u_list[layer] = self.act(spmm(sparse_dropout(self.adj_norm,self.dropout), self.E_i_list[layer-1], self.device))
                self.Z_i_list[layer] = self.act(spmm(sparse_dropout(self.adj_norm,self.dropout).transpose(0,1), self.E_u_list[layer-1], self.device))
                
                # svd_adj propagation
                vt_ei = self.vt @ self.E_i_list[layer-1]
                self.G_u_list[layer] = self.act(self.u_mul_s @ vt_ei)
                ut_eu = self.ut @ self.E_u_list[layer-1]
                self.G_i_list[layer] = self.act(self.v_mul_s @ ut_eu)

                # aggregate
                self.E_u_list[layer] = self.Z_u_list[layer] + self.E_u_list[layer-1]
                self.E_i_list[layer] = self.Z_i_list[layer] + self.E_i_list[layer-1]

            # aggregate across layers
            self.E_u = sum(self.E_u_list) # User embeddings
            self.E_i = sum(self.E_i_list) # Item embeddings

            # cl loss
            loss_s = 0
            for l in range(1,self.l+1):
                u_mask = (torch.rand(len(uids))>0.5).float().to(self.device)

                gnn_u = nn.functional.normalize(self.Z_u_list[l][uids],p=2,dim=1)
                hyper_u = nn.functional.normalize(self.G_u_list[l][uids],p=2,dim=1)
                hyper_u = self.Ws[l-1](hyper_u)
                pos_score = torch.exp((gnn_u*hyper_u).sum(1)/self.temp)
                neg_score = torch.exp(gnn_u @ hyper_u.T/self.temp).sum(1)
                loss_s_u = ((-1 * torch.log(pos_score/(neg_score+1e-8) + 1e-8))*u_mask).sum()
                loss_s = loss_s + loss_s_u

                i_mask = (torch.rand(len(iids))>0.5).float().to(self.device)

                gnn_i = nn.functional.normalize(self.Z_i_list[l][iids],p=2,dim=1)
                hyper_i = nn.functional.normalize(self.G_i_list[l][iids],p=2,dim=1)
                hyper_i = self.Ws[l-1](hyper_i)
                pos_score = torch.exp((gnn_i*hyper_i).sum(1)/self.temp)
                neg_score = torch.exp(gnn_i @ hyper_i.T/self.temp).sum(1)
                loss_s_i = ((-1 * torch.log(pos_score/(neg_score+1e-8) + 1e-8))*i_mask).sum()
                loss_s = loss_s + loss_s_i
            
            # bpr loss
            loss_r = 0
            for i in range(len(uids)):
                u = uids[i]
                u_emb = self.E_u[u]
                u_pos = pos[i]
                u_neg = neg[i]
                pos_emb = self.E_i[u_pos]
                neg_emb = self.E_i[u_neg]
                pos_scores = u_emb @ pos_emb.T
                neg_scores = u_emb @ neg_emb.T
                bpr = nn.functional.relu(1-pos_scores+neg_scores)
                loss_r = loss_r + bpr.sum()
            loss_r = loss_r/self.batch_user
            
            # msbe l_sim loss
            loss_msbe = torch.tensor(0.0).to(self.device)
            if self.msb_rate > 0:
                sim_uids = self.get_msb_samples(uids, self.user_neighbors)
                loss_msbe_u = InfoNCE(self.E_u[uids], self.E_u[sim_uids], self.tau)
                
                sim_iids = self.get_msb_samples(iids, self.item_neighbors)
                loss_msbe_i = InfoNCE(self.E_i[iids], self.E_i[sim_iids], self.tau)
                
                loss_msbe = loss_msbe_u + loss_msbe_i

            # Total loss calculation (Added)
            loss = loss_r + self.lambda_1 * loss_s + self.msb_rate * loss_msbe

            return loss, loss_r, loss_s, loss_msbe

# ---------------- MAIN TRAINING LOOP ----------------

# hyperparameters
d = args.d
l = args.gnn_layer
temp = args.temp
batch_user = args.batch
epoch_no = args.epoch
max_samp = 40
lambda_1 = args.lambda1
lambda_2 = args.lambda2
dropout = args.dropout
lr = args.lr
svd_q = args.q
msb_rate = args.msb_rate
sim_threshold = args.sim_threshold

# load data
path = 'data/' + args.data + '/'
if not os.path.exists(path):
    print(f"Data path {path} not found. Trying to clone again or verify...")
    if not os.path.exists('data'):
         print("Critical: 'data' extraction failed.")
else:
    print(f"Loading data from {path}")

f = open(path+'trnMat.pkl','rb')
train = pickle.load(f)
train_csr = (train!=0).astype(np.float32)
f = open(path+'tstMat.pkl','rb')
test = pickle.load(f)
print('Data loaded.')

print('user_num:',train.shape[0],'item_num:',train.shape[1],'lambda_1:',lambda_1,'lambda_2:',lambda_2,'temp:',temp,'q:',svd_q)

# Load MSBE Neighbors
user_neighbors, item_neighbors = load_msbe_neighbors(train, sim_threshold=sim_threshold)


epoch_user = min(train.shape[0], 30000)

adj = scipy_sparse_mat_to_torch_sparse_tensor(train).coalesce().to(device)

print('Performing SVD...')
svd_u,s,svd_v = torch.svd_lowrank(adj,q=svd_q)
u_mul_s = svd_u @ torch.diag(s)
v_mul_s = svd_v @ torch.diag(s)
del adj
del s
print('SVD done.')

rowD = np.array(train.sum(1)).squeeze()
colD = np.array(train.sum(0)).squeeze()
for i in range(len(train.data)):
    train.data[i] = train.data[i] / pow(rowD[train.row[i]]*colD[train.col[i]], 0.5)
adj_norm = scipy_sparse_mat_to_torch_sparse_tensor(train)
adj_norm = adj_norm.coalesce().to(device)
print('Adj matrix normalized.')

test_labels = [[] for i in range(test.shape[0])]
for i in range(len(test.data)):
    row = test.row[i]
    col = test.col[i]
    test_labels[row].append(col)
print('Test data processed.')

loss_list = []
loss_r_list = []
loss_s_list = []
loss_msbe_list = []
recall_20_x = []
recall_20_y = []
ndcg_20_y = []
recall_40_y = []
ndcg_40_y = []

# Initialize model
model = LightGCL(adj_norm.shape[0], adj_norm.shape[1], d, u_mul_s, v_mul_s, svd_u.T, svd_v.T, train_csr, adj_norm, l, temp, lambda_1, dropout, batch_user, device, user_neighbors, item_neighbors, msb_rate)
model.to(device)
# Note: old_setting uses weight_decay=lambda_2
optimizer = torch.optim.Adam(model.parameters(),weight_decay=lambda_2,lr=lr)

def learning_rate_decay(optimizer):
    for param_group in optimizer.param_groups:
        lr = param_group['lr']*0.98
        if lr > 0.0005:
            param_group['lr'] = lr
    return lr

current_lr = lr

for epoch in range(epoch_no):
    if (epoch+1)%50 == 0:
        if not os.path.exists('saved_model'): os.makedirs('saved_model')
        torch.save(model.state_dict(),'saved_model/saved_model_epoch_'+str(epoch)+'.pt')
        torch.save(optimizer.state_dict(),'saved_model/saved_optim_epoch_'+str(epoch)+'.pt')

    current_lr = learning_rate_decay(optimizer)
        
    e_users = np.random.permutation(adj_norm.shape[0])[:epoch_user]
    batch_no = int(np.ceil(epoch_user/batch_user))

    epoch_loss = 0
    epoch_loss_r = 0
    epoch_loss_s = 0
    epoch_loss_msbe = 0
    
    # Manual batch loop (no tqdm to avoid verbose output if preferred, or keep simplicitly)
    for batch in range(batch_no):
        start = batch*batch_user
        end = min((batch+1)*batch_user,epoch_user)
        batch_users = e_users[start:end]

        # sample pos and neg
        pos = []
        neg = []
        iids = set()
        for i in range(len(batch_users)):
            u = batch_users[i]
            # Handle potential sparse format differences if train_csr behaves differently
            # In old_setting, train_csr is scipy.sparse.csr_matrix
            u_interact = train_csr[u].toarray()[0]
            positive_items = np.random.permutation(np.where(u_interact==1)[0])
            negative_items = np.random.permutation(np.where(u_interact==0)[0])
            item_num = min(max_samp,len(positive_items))
            positive_items = positive_items[:item_num]
            negative_items = negative_items[:item_num]
            pos.append(torch.LongTensor(positive_items).to(device))
            neg.append(torch.LongTensor(negative_items).to(device))
            iids = iids.union(set(positive_items))
            iids = iids.union(set(negative_items))
        iids = torch.LongTensor(list(iids)).to(device)
        uids = torch.LongTensor(batch_users).to(device)

        # feed
        optimizer.zero_grad()
        loss, loss_r, loss_s, loss_msbe = model(uids, iids, pos, neg)
        loss.backward()
        optimizer.step()

        torch.cuda.empty_cache()

        epoch_loss += loss.cpu().item()
        epoch_loss_r += loss_r.cpu().item()
        epoch_loss_s += loss_s.cpu().item()
        epoch_loss_msbe += loss_msbe.cpu().item()

    epoch_loss = epoch_loss/batch_no
    epoch_loss_r = epoch_loss_r/batch_no
    epoch_loss_s = epoch_loss_s/batch_no
    epoch_loss_msbe = epoch_loss_msbe/batch_no
    
    loss_list.append(epoch_loss)
    loss_r_list.append(epoch_loss_r)
    loss_s_list.append(epoch_loss_s)
    loss_msbe_list.append(epoch_loss_msbe)
    
    print(f'Epoch: {epoch} Loss: {epoch_loss:.4f} Loss_r: {epoch_loss_r:.4f} Loss_s: {epoch_loss_s:.4f} Loss_msbe: {epoch_loss_msbe:.4f}')

    if epoch % 1 == 0:  # test every epoch
        test_uids = np.array([i for i in range(adj_norm.shape[0])])
        batch_no = int(np.ceil(len(test_uids)/batch_user))

        all_recall_20 = 0
        all_ndcg_20 = 0
        all_recall_40 = 0
        all_ndcg_40 = 0
        for batch in range(batch_no):
            start = batch*batch_user
            end = min((batch+1)*batch_user,len(test_uids))

            test_uids_input = torch.LongTensor(test_uids[start:end]).to(device)
            predictions = model(test_uids_input,None,None,None,test=True)
            predictions = np.array(predictions.cpu())

            #top@20
            recall_20, ndcg_20 = metrics(test_uids[start:end],predictions,20,test_labels)
            #top@40
            recall_40, ndcg_40 = metrics(test_uids[start:end],predictions,40,test_labels)

            all_recall_20+=recall_20
            all_ndcg_20+=ndcg_20
            all_recall_40+=recall_40
            all_ndcg_40+=ndcg_40
        print('-------------------------------------------')
        print('Test of epoch',epoch,':','Recall@20:',all_recall_20/batch_no,'Ndcg@20:',all_ndcg_20/batch_no,'Recall@40:',all_recall_40/batch_no,'Ndcg@40:',all_ndcg_40/batch_no)
        recall_20_x.append(epoch)
        recall_20_y.append(all_recall_20/batch_no)
        ndcg_20_y.append(all_ndcg_20/batch_no)
        recall_40_y.append(all_recall_40/batch_no)
        ndcg_40_y.append(all_ndcg_40/batch_no)

# final test
test_uids = np.array([i for i in range(adj_norm.shape[0])])
batch_no = int(np.ceil(len(test_uids)/batch_user))

all_recall_20 = 0
all_ndcg_20 = 0
all_recall_40 = 0
all_ndcg_40 = 0
for batch in range(batch_no):
    start = batch*batch_user
    end = min((batch+1)*batch_user,len(test_uids))

    test_uids_input = torch.LongTensor(test_uids[start:end]).to(device)
    predictions = model(test_uids_input,None,None,None,test=True)
    predictions = np.array(predictions.cpu())

    #top@20
    recall_20, ndcg_20 = metrics(test_uids[start:end],predictions,20,test_labels)
    #top@40
    recall_40, ndcg_40 = metrics(test_uids[start:end],predictions,40,test_labels)

    all_recall_20+=recall_20
    all_ndcg_20+=ndcg_20
    all_recall_40+=recall_40
    all_ndcg_40+=ndcg_40

print('-------------------------------------------')
print('Final test:','Recall@20:',all_recall_20/batch_no,'Ndcg@20:',all_ndcg_20/batch_no,'Recall@40:',all_recall_40/batch_no,'Ndcg@40:',all_ndcg_40/batch_no)

recall_20_x.append('Final')
recall_20_y.append(all_recall_20/batch_no)
ndcg_20_y.append(all_ndcg_20/batch_no)
recall_40_y.append(all_recall_40/batch_no)
ndcg_40_y.append(all_ndcg_40/batch_no)

metric = pd.DataFrame({
    'epoch':recall_20_x,
    'recall@20':recall_20_y,
    'ndcg@20':ndcg_20_y,
    'recall@40':recall_40_y,
    'ndcg@40':ndcg_40_y
})
current_t = time.gmtime()
if not os.path.exists('log'):
    os.makedirs('log')

metric.to_csv('log/result_'+args.data+'_'+time.strftime('%Y-%m-%d_%H_%M_%S',current_t)+'.csv')

if not os.path.exists('saved_model'):
    os.makedirs('saved_model')
torch.save(model.state_dict(),'saved_model/saved_model_'+args.data+'_'+time.strftime('%Y-%m-%d_%H_%M_%S',current_t)+'.pt')
torch.save(optimizer.state_dict(),'saved_model/saved_optim_'+args.data+'_'+time.strftime('%Y-%m-%d_%H_%M_%S',current_t)+'.pt')